# Lesson 12 — Interior-Point Methods

## Learning objectives

1. State the path-following IPM for LP.
2. Walk through one Mehrotra predictor-corrector step {cite:p}`Mehrotra1992`.
3. Compare IPM vs simplex on tall, fat, and dense LPs.
4. Recognize crossover (turning IPM solution into a basic solution).

## 1. The KKT system for LP

For $\min c^\top x$ s.t. $Ax = b, x \ge 0$, the KKT conditions are

$$Ax = b, \quad A^\top y + s = c, \quad x_i s_i = 0, \quad x, s \ge 0.$$

Replace the complementarity $x_i s_i = 0$ with $x_i s_i = \mu$ for a *barrier parameter* $\mu > 0$. Solve this perturbed system, then drive $\mu \to 0$. The locus of solutions is the **central path** {cite:p}`Wright1997`.

## 2. Newton step on the perturbed KKT

$$\begin{bmatrix} 0 & A^\top & I \\ A & 0 & 0 \\ S & 0 & X \end{bmatrix} \begin{bmatrix} \Delta x \\ \Delta y \\ \Delta s \end{bmatrix} = \begin{bmatrix} c - A^\top y - s \\ b - Ax \\ \mu e - X S e \end{bmatrix}$$

with $X = \text{diag}(x), S = \text{diag}(s), e = (1, ..., 1)^\top$. Eliminate $\Delta s$ to get a smaller normal-equation system $A D A^\top \Delta y = r$ with $D = X S^{-1}$ — solve via Cholesky.

## 3. Mehrotra's predictor-corrector

A practical IPM {cite:p}`Mehrotra1992`:

1. **Predictor:** solve the linear system with $\mu = 0$ on the right-hand side (the affine-scaling direction $\Delta^{\text{aff}}$). Compute the maximal step $\alpha^{\text{aff}} \in (0, 1]$ keeping $x, s$ positive, and the post-step duality measure $\mu_{\text{aff}} = (x + \alpha^{\text{aff}} \Delta x^{\text{aff}})^\top (s + \alpha^{\text{aff}} \Delta s^{\text{aff}}) / n$.
2. **Centering parameter:** $\sigma = (\mu_{\text{aff}} / \mu_{\text{cur}})^3$ — the cube reflects "if the affine step works well, centre less; if it doesn't, centre more". The new target is $\mu_{\text{new}} = \sigma \, \mu_{\text{cur}}$.
3. **Corrector:** re-solve with right-hand side $\sigma \mu_{\text{cur}} e - X S e - \Delta X^{\text{aff}} \Delta S^{\text{aff}} e$, picking up second-order centring terms.

The resulting algorithm typically converges in 20–50 iterations regardless of problem size.

## 4. Crossover

IPM converges to an *interior* solution near the central path. To get a *basic* (vertex) solution (which simplex returns natively), run a *crossover* phase: a small simplex run starting from the IPM iterate. Many users want crossover for warm-starting subsequent solves; some don't care.

`discopt`'s stable solve API does not let you toggle simplex-vs-IPM-vs-crossover from Python — the Rust backend chooses. To *measure* the trade-off in the exercises below you'll use `scipy.optimize.linprog` (which exposes `method="highs-ds"` for simplex and `method="highs-ipm"` for IPM), or roll your own toy IPM in pure Python.

In [ ]:
# Standard discopt course imports. The lessons target the real
# `discopt.modeling.core.Model` API: `m.continuous(name, shape=..., lb=..., ub=...)`,
# `m.binary(name, shape=...)`, `m.integer(name, shape=..., lb=..., ub=...)`,
# `m.subject_to(...)`, `m.minimize(...) / .maximize(...)`, `m.solve(...)`.
# Result attributes: `r.status`, `r.objective`, `r.gap`, `r.bound`,
# `r.wall_time`, `r.node_count`, and `r.value(var)` for variable arrays.
import numpy as np
import discopt as do
import discopt.modeling as dm


In [ ]:
import numpy as np, discopt as do, time

# Generate a tall, dense LP (a regime where IPM-style methods shine).
rng = np.random.default_rng(0)
m_, n_ = 1000, 200
A = rng.normal(size=(m_, n_))
b = A @ np.abs(rng.normal(size=n_)) + np.abs(rng.normal(size=m_))
c = rng.normal(size=n_)

mod = do.Model("tall")
x = mod.continuous("x", shape=(n_,), lb=0)
mod.subject_to(A @ x <= b)
mod.minimize(c @ x)

t = time.time(); r = mod.solve(); dt = time.time() - t
print(f"objective = {r.objective:.6f}   wall_time = {dt:.2f}s   status = {r.status}")
# `discopt.solve()` does not let you choose simplex vs IPM at the public
# API level; the Rust backend picks the appropriate path. Exercises below
# implement a toy IPM in pure Python so you can see the iteration counts.


## 5. When IPM beats simplex (and vice versa)

| Structure | Winner |
|-----------|--------|
| Sparse, well-conditioned, "few" constraints active | simplex |
| Dense, many constraints, high dim | IPM |
| Re-solving with small perturbation | simplex (warm start) |
| First solve, large LP | IPM |
| Need basic solution | simplex (or IPM + crossover) |

This trade-off is the practical flip side of the polynomial-vs-exponential theory.

## References
{cite:p}`Karmarkar1984` (the breakthrough), {cite:p}`Mehrotra1992` (practical), {cite:p}`Wright1997` (textbook), {cite:p}`Khachiyan1979` (the earlier ellipsoid result).